# Test based on https://github.com/NathanWycoff/prox_alasso/blob/main/python/ada_sim.py

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from adaptivelasso.skcompat import MAPAdaptiveLasso, MAPAdaptiveLassoCV
from adaptivelasso.utils import generate_data, report_model_results
import numpy as np
from sklearn.datasets import fetch_openml
import time
from sklearn.linear_model import LogisticRegressionCV
import pandas as pd


## Compare MAPAdaptiveLasso with LassoCV

In [3]:
# Example for normal likelihood
N, P = 1000, 20
beta = np.zeros(P)
beta[:5] = 2  # 5 informative features
sigma = 1.0
lik = 'normal'

X, y = generate_data(lik, N, P, beta=beta, sigma=sigma, random_state=42)

# Split and fit as before...
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=5000, track=False, tol=1e-4, cv=5))
])
pipe.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe.named_steps['jax_model'], X_test, y_test, lik)

# Use a pipeline for scaling and LassoCV
pipe_lassocv = Pipeline([
    ('scale', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=0, max_iter=5000, tol=1e-4))
])
pipe_lassocv.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe_lassocv.named_steps['lasso'], X_test, y_test, lik=lik)

ES stop!
ES stop!
ES stop!
ES stop!
ES stop!
Estimated coefficients: [ 1.99321621  2.04238974  2.05095199  1.99129659  1.95324436  0.00745593
  0.05981441  0.          0.02196742 -0.01425156  0.00656693 -0.02367048
 -0.02141572 -0.01224012  0.04273658  0.0606819   0.0600932   0.01857392
 -0.01588552  0.09296891]
R²: 0.9523
RMSE: 0.9496
Estimated error variance: 0.9015898672894598
Estimated coefficients: [ 1.96264089  2.01882337  2.0179528   1.93265903  1.90844035  0.01096407
  0.03442297 -0.          0.         -0.         -0.         -0.
  0.          0.          0.0219134   0.00486638  0.          0.
 -0.          0.0357031 ]
R²: 0.9528
RMSE: 0.9444
Estimated error variance: 0.8911630396832502


{'coefs': array([ 1.96264089,  2.01882337,  2.0179528 ,  1.93265903,  1.90844035,
         0.01096407,  0.03442297, -0.        ,  0.        , -0.        ,
        -0.        , -0.        ,  0.        ,  0.        ,  0.0219134 ,
         0.00486638,  0.        ,  0.        , -0.        ,  0.0357031 ]),
 'R2': 0.9527899072830069,
 'RMSE': np.float64(0.9444386366702587),
 'MAD': None}

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from adaptivelasso.skcompat import MAPAdaptiveLasso

# Assume generate_data and report_model_results are already defined as above

likelihoods = ['normal', 'cauchy', 'bernoulli', 'poisson', 'zinb', 'nb']
results = []
max_iters = 3000  # Set a maximum number of iterations
tol = 1e-4  # Set a tolerance for convergence
cv = 5

for lik in likelihoods:
    print(f"Testing likelihood: {lik}")
    N, P = 1000, 20
    true_coefs = np.zeros(P)
    true_coefs[:3] = 1.0
    sigma = 1.0

    X, y = generate_data(lik, N, P, beta=true_coefs, sigma=sigma, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

    # MAPAdaptiveLasso
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=max_iters, track=True, tol=tol, cv=cv))
    ])
    pipe.fit(X_train, y_train)
    adapt_metrics = report_model_results(pipe.named_steps['jax_model'], X_test, y_test, lik)

    # LassoCV
    pipe_lasso = Pipeline([
        ('scale', StandardScaler()),
        ('lasso', LassoCV(cv=5, random_state=0, max_iter=max_iters))
    ])
    pipe_lasso.fit(X_train, y_train)
    lasso_metrics = report_model_results(pipe_lasso.named_steps['lasso'], X_test, y_test, lik)

    # Decide which is better for each metric
    if lik == 'cauchy':
        better_mad = (
            'AdaptiveLasso' if adapt_metrics['MAD'] < lasso_metrics['MAD'] else
            'LassoCV' if lasso_metrics['MAD'] < adapt_metrics['MAD'] else
            'Tie'
        )
        better_rmse = np.nan
    else:
        better_mad = np.nan
        better_rmse = (
            'AdaptiveLasso' if adapt_metrics['RMSE'] < lasso_metrics['RMSE'] else
            'LassoCV' if lasso_metrics['RMSE'] < adapt_metrics['RMSE'] else
            'Tie'
        )

    if np.isnan(lasso_metrics['R2']):
        better_r2 = 'AdaptiveLasso only'
    else:
        better_r2 = (
            'AdaptiveLasso' if adapt_metrics['R2'] > lasso_metrics['R2'] else
            'LassoCV' if lasso_metrics['R2'] > adapt_metrics['R2'] else
            'Tie'
        )

    results.append({
        'likelihood': lik,
        'R2_AdaptiveLasso': adapt_metrics['R2'],
        'RMSE_AdaptiveLasso': adapt_metrics['RMSE'],
        'MAD_AdaptiveLasso': adapt_metrics['MAD'],
        'R2_LassoCV': lasso_metrics['R2'],
        'RMSE_LassoCV': lasso_metrics['RMSE'],
        'MAD_LassoCV': lasso_metrics['MAD'],
        'Better_R2': better_r2,
        'Better_RMSE': better_rmse,
        'Better_MAD': better_mad,
    })

df_results = pd.DataFrame(results)
print(df_results)

Testing likelihood: normal
ES stop!
ES stop!
ES stop!
ES stop!
Estimated coefficients: [ 1.04177314  1.06506323  1.00960306 -0.01622986 -0.01914712  0.02094688
  0.07715846  0.00134746 -0.01181379 -0.02115135  0.03019611 -0.048533
  0.00401244  0.03012038  0.02391426  0.06165883 -0.00468517  0.01157192
 -0.0174364   0.05880315]
R²: 0.7328
RMSE: 0.9434
Estimated error variance: 0.889046835808645
Estimated coefficients: [ 0.95982998  0.99692659  1.00861647 -0.         -0.          0.
  0.02881936 -0.          0.         -0.         -0.         -0.
 -0.          0.          0.00593973  0.          0.          0.
 -0.          0.02557177]
R²: 0.7370
RMSE: 0.9360
Estimated error variance: 0.8745377131674651
Testing likelihood: cauchy
ES stop!
ES stop!
ES stop!
Estimated coefficients: [ 1.04944012  1.05288893  0.89940316  0.0857037  -0.0947982   0.050131
 -0.03410379 -0.02627072 -0.04953316 -0.06135845 -0.          0.03783665
  0.01171731 -0.08662611 -0.01989594 -0.08019521 -0.14025776 -0.04

### spambase

In [3]:
X, y = fetch_openml(name="spambase", version=1, as_frame=False, return_X_y=True)
y = y.astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
param_grid = {
    'tau0': [0.1,0.2,0.3],
    'lr_pre': [0.01],
    'max_iters': [3000],
    'lik': ['bernoulli']  
}
# Use a pipeline for scaling and MAPAdaptiveLassoCV
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('alasso_cv', MAPAdaptiveLassoCV(param_grid=param_grid, n_splits=5, scoring='roc_auc', random_state=42, verbose=True))
])
start_time = time.time()
pipe.fit(X_train, y_train)
end_time = time.time()
# Report results
metrics = report_model_results(pipe.named_steps['alasso_cv'], X_test, y_test, lik='bernoulli')
print("MAPAdaptiveLassoCV test metrics:", metrics)
print(f"MAPAdaptiveLassoCV Training Time: {end_time - start_time:.2f} seconds")

# Use a pipeline for scaling and LogisticRegressionCV 
pipe_lassocv = Pipeline([
    ('scale', StandardScaler()),
    ('lasso', LogisticRegressionCV(cv=5, random_state=0, max_iter=3000, tol=1e-4))
])
start_time = time.time()
pipe_lassocv.fit(X_train, y_train)
end_time = time.time()
# Report results
report_model_results(pipe_lassocv.named_steps['lasso'], X_test, y_test, lik='bernoulli')
print(f"LogisticRegressionCV Training Time: {end_time - start_time:.2f} seconds")


Testing params: {'lik': 'bernoulli', 'lr_pre': 0.01, 'max_iters': 3000, 'tau0': 0.1}
Testing params: {'lik': 'bernoulli', 'lr_pre': 0.01, 'max_iters': 3000, 'tau0': 0.2}
Testing params: {'lik': 'bernoulli', 'lr_pre': 0.01, 'max_iters': 3000, 'tau0': 0.3}
Estimated coefficients: [-0.00543899 -0.09365772  0.10155387  0.18533882  0.33314285  0.18742157
  0.78113757  0.25473674  0.18487628  0.07196485  0.17821195 -0.12356905
  0.00361137  0.1691086   0.22028289  0.41092366  0.34991818  0.17802869
  0.14866985  0.29114811  0.35004743  0.26794792  0.65923138  0.33004719
 -0.54125416 -0.34808192 -0.44777434  0.00341595 -0.21844946 -0.22184289
 -0.10118994 -0.06085801 -0.25149129 -0.06019825 -0.19953831  0.0750016
 -0.17902688 -0.05312779 -0.19304059 -0.01701139 -0.18487496 -0.34800303
 -0.13645424 -0.23868276 -0.39466437 -0.37479485 -0.13700464 -0.22294911
 -0.22416273 -0.05296789 -0.08760705  0.33364752  0.74248989  0.19009207
  0.16297199  0.3493001   0.39405155]
Accuracy: 0.4452
AUC: 0.750

### UCI Bike Sharing Dataset

In [6]:
#Download via http://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip
df = pd.read_csv('/Users/linpei/Downloads/Bike-Sharing-Dataset/day.csv')
X = df.drop(columns=['instant', 'dteday', 'casual', 'registered', 'cnt'])
y = df['cnt']
# One-hot Encode Categorical Variables
X = pd.get_dummies(X, columns=['season', 'mnth', 'weekday', 'weathersit'], drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

param_grid = {
    'tau0': [0.1,0.2, 0.3],
    'lr_pre': [0.01, 0.1],
    'max_iters': [3000],
    'lik': ['nb'] 

}

model_cv = MAPAdaptiveLassoCV(param_grid=param_grid, n_splits=5, scoring='neg_log_likelihood', verbose=True)
start_time = time.time()
model_cv.fit(X_train.values, y_train.values)
end_time = time.time()
# Report results
metrics = report_model_results(model_cv, X_test, y_test, lik='nb')
print("MAPAdaptiveLassoCV test metrics:", metrics)
print(f"MAPAdaptiveLassoCV Training Time: {end_time - start_time:.2f}s")


lasso = LassoCV(cv=5, max_iter=3000, n_jobs=-1)
start_time = time.time()
lasso.fit(X_train, y_train)
end_time = time.time()
# Report results
report_model_results(lasso, X_test, y_test, lik='nb')
print("LogisticRegressionCV test metrics:", metrics)
print(f"LogisticRegressionCV Training Time: {end_time - start_time:.2f} seconds")

Testing params: {'lik': 'nb', 'lr_pre': 0.01, 'max_iters': 3000, 'tau0': 0.1}
ES stop!
ES stop!
ES stop!
ES stop!
Testing params: {'lik': 'nb', 'lr_pre': 0.01, 'max_iters': 3000, 'tau0': 0.2}
ES stop!
ES stop!
ES stop!
ES stop!
Testing params: {'lik': 'nb', 'lr_pre': 0.01, 'max_iters': 3000, 'tau0': 0.3}
ES stop!
ES stop!
ES stop!
Testing params: {'lik': 'nb', 'lr_pre': 0.1, 'max_iters': 3000, 'tau0': 0.1}
ES stop!
ES stop!
ES stop!
ES stop!
Testing params: {'lik': 'nb', 'lr_pre': 0.1, 'max_iters': 3000, 'tau0': 0.2}
ES stop!
ES stop!
ES stop!
ES stop!
Testing params: {'lik': 'nb', 'lr_pre': 0.1, 'max_iters': 3000, 'tau0': 0.3}
ES stop!
ES stop!
ES stop!
ES stop!
ES stop!
ES stop!
Estimated coefficients: [ 0.22405522 -0.01762567  0.0254484   0.15601611  0.15461604 -0.07219477
 -0.05551203  0.08856771  0.05357176  0.159175    0.01469401  0.04602898
  0.0283804   0.04369431  0.00852344 -0.00751429  0.01937851  0.06317776
  0.04476937  0.03510748  0.02407974 -0.00238091 -0.         -0.003

/Users/linpei/miniforge3/envs/tf_m1/lib/python3.8/site-packages/sklearn/base.py:443: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
